# 06 Security Posture Checks for Generated Changes (Agent-CLIs, 2026)

## What This Lesson Is
Detect risky generated code patterns before merge using deterministic checks.

## Scientific Lens
- Concept: Pre-merge static security screening
- Measure: Detection coverage for secret and unsafe execution patterns
- Validity Limit: Pattern checks need complementary semantic analysis to reduce false positives/negatives.


## How It Works
1. Run deterministic diff scan.
2. Categorize findings by severity.
3. Scan live generated output from CLI when available.


In [ ]:
print("Security posture lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
import re

lines = [
    '+ OPENAI_API_KEY="sk-live-abc123"',
    '+ subprocess.run("curl http://example", shell=True)',
    '+ SAFE_SETTING=true',
]

secret_re = re.compile(r"sk-[A-Za-z0-9-]+")
unsafe_shell_re = re.compile(r"shell=True")

findings = []
for line in lines:
    if secret_re.search(line):
        findings.append(("secret", line))
    if unsafe_shell_re.search(line):
        findings.append(("unsafe_shell", line))

print(findings)
assert len(findings) == 2


In [ ]:
# Live Demo
import re
import shutil
import subprocess

prompt = "Output a tiny Python snippet that validates JSON and avoids unsafe shell execution."
cmds = [
    ["openclaw", "agent", "--local", "--to", "+15555550123", "--message", prompt, "--timeout", "70"],
    ["codex", "exec", prompt],
    ["claude", "-p", prompt],
    ["opencode", "run", prompt],
]

for cmd in cmds:
    if shutil.which(cmd[0]) is None:
        continue
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if p.returncode != 0:
        continue
    text = (p.stdout or p.stderr).strip()
    print(text[:1200])
    print("contains shell=True:", bool(re.search(r"shell=True", text)))
    break
else:
    print("Skipping live security scan: no compatible non-interactive CLI available.")


## Applied Labs
1. Add detection rules for hardcoded cloud credentials and private keys.
2. Map findings to severity tiers and block thresholds.
3. Integrate scanner output into a machine-readable JSON report.

## Validation Checklist
- Scanner detects at least secret leakage and unsafe shell usage.
- Findings include enough context to remediate quickly.
- Live output can be scanned by same deterministic rules.

## Further Reading
- [OWASP Secure Coding Practices](https://owasp.org/www-project-secure-coding-practices-quick-reference-guide/)
- [GitHub Secret Scanning Patterns](https://docs.github.com/code-security/secret-scanning)
- [Bandit Python Security Linter](https://bandit.readthedocs.io/)
